# EDA Analysis

Exploratory analysis for mutual fund NAV, AUM, SIP, investor, geography, folio, correlation, and portfolio allocation trends.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
CHART_DIR = PROJECT_ROOT / 'reports' / 'charts' / 'eda'
sns.set_theme(style='whitegrid')

In [ ]:
fund = pd.read_csv(PROCESSED_DIR / 'fund_master_cleaned.csv', parse_dates=['launch_date'])
nav = pd.read_csv(PROCESSED_DIR / 'nav_history_cleaned.csv', parse_dates=['date'])
aum = pd.read_csv(PROCESSED_DIR / 'aum_by_fund_house_cleaned.csv', parse_dates=['date'])
sip = pd.read_csv(PROCESSED_DIR / 'monthly_sip_inflows_cleaned.csv', parse_dates=['month'])
category = pd.read_csv(PROCESSED_DIR / 'category_inflows_cleaned.csv', parse_dates=['month'])
folios = pd.read_csv(PROCESSED_DIR / 'industry_folio_count_cleaned.csv', parse_dates=['month'])
performance = pd.read_csv(PROCESSED_DIR / 'scheme_performance_cleaned.csv')
tx = pd.read_csv(PROCESSED_DIR / 'investor_transactions_cleaned.csv', parse_dates=['transaction_date'])
holdings = pd.read_csv(PROCESSED_DIR / 'portfolio_holdings_cleaned.csv', parse_dates=['portfolio_date'])
nav = nav.merge(fund[['amfi_code', 'scheme_name', 'fund_house']], on='amfi_code', how='left')

Insight 1: Chart 01 shows broad NAV participation across schemes, with the 2023 rally and 2024 correction windows visible as market-wide regimes.

In [ ]:
fig = px.line(nav, x='date', y='nav', color='scheme_name', title='Daily NAV Trend for All Schemes')
fig.add_vrect(x0='2023-01-01', x1='2023-12-31', fillcolor='green', opacity=0.12, line_width=0)
fig.add_vrect(x0='2024-01-01', x1='2024-12-31', fillcolor='red', opacity=0.10, line_width=0)
fig.show()

Insight 2: Chart 02 highlights that large fund houses dominate AUM, with SBI marked against the 12.5 lakh crore reference level.

In [ ]:
aum.assign(year=aum['date'].dt.year).pivot_table(index='year', columns='fund_house', values='aum_lakh_crore').plot(kind='bar', figsize=(14, 6)); plt.title('AUM Growth by Fund House'); plt.ylabel('AUM, lakh crore');

Insight 3: Chart 03 shows SIP inflows compounding strongly into the late-2025 peak.

In [ ]:
px.line(sip, x='month', y='sip_inflow_crore', markers=True, title='Monthly SIP Inflows').show()

Insight 4: Chart 04 reveals category-level rotation through varying net inflow intensity by month.

In [ ]:
category.assign(month_label=category['month'].dt.strftime('%Y-%m')).pivot_table(index='category', columns='month_label', values='net_inflow_crore').pipe(lambda x: sns.heatmap(x, cmap='YlGnBu')); plt.title('Category Inflow Heatmap');

Insight 5: Charts 05-07 show investor mix across age, SIP ticket size, and gender.

In [ ]:
tx['age_group'].value_counts().sort_index().plot(kind='pie', autopct='%1.1f%%', figsize=(6, 6)); plt.title('Age Group Distribution');

Insight 6: Charts 08-09 show geographic contribution, separating state-level SIP amount from T30/B30 city-tier concentration.

In [ ]:
tx[tx.transaction_type == 'SIP'].groupby('state')['amount_inr'].sum().sort_values().tail(15).plot(kind='barh', figsize=(9, 6)); plt.title('SIP Amount by State');

Insight 7: Chart 10 tracks industry folio growth from the January 2022 base to the December 2025 endpoint.

In [ ]:
sns.lineplot(data=folios, x='month', y='total_folios_crore', marker='o'); plt.title('Folio Count Growth');

Insight 8: Chart 11 shows that large equity funds have mostly positive return correlations, supporting common market-factor exposure.

In [ ]:
selected = performance.nlargest(10, 'aum_crore')['amfi_code']; corr = nav[nav.amfi_code.isin(selected)].pivot(index='date', columns='scheme_name', values='nav').pct_change(fill_method=None).corr(); sns.heatmap(corr, cmap='vlag', center=0); plt.title('Return Correlation Matrix');

Insight 9: Chart 12 summarizes aggregate sector exposure across equity portfolio holdings.

In [ ]:
holdings.groupby('sector')['weight_pct'].sum().sort_values(ascending=False).plot(kind='pie', wedgeprops={'width': .42}, figsize=(7, 7)); plt.title('Sector Allocation Donut');

Insight 10: Charts 13-15 connect performance, cost, and transaction behavior for fund comparison and investor activity review.

In [ ]:
performance.sort_values('return_3yr_pct').tail(10).plot(kind='barh', x='scheme_name', y='return_3yr_pct', legend=False, figsize=(9, 6)); plt.title('Top 10 Funds by 3-Year Return');

## Final Report Chart Gallery

![01 NAV trend](../reports/charts/eda/01_nav_trend_all_schemes.png)

![02 AUM growth](../reports/charts/eda/02_aum_growth_by_fund_house.png)

![03 SIP inflow](../reports/charts/eda/03_sip_inflow_time_series.png)

![04 Category heatmap](../reports/charts/eda/04_category_inflow_heatmap.png)

![05 Age distribution](../reports/charts/eda/05_age_group_distribution.png)

![06 SIP by age](../reports/charts/eda/06_sip_amount_by_age_group.png)

![07 Gender split](../reports/charts/eda/07_gender_split.png)

![08 SIP by state](../reports/charts/eda/08_sip_amount_by_state.png)

![09 City tier](../reports/charts/eda/09_city_tier_split.png)

![10 Folio growth](../reports/charts/eda/10_folio_count_growth.png)

![11 Return correlation](../reports/charts/eda/11_nav_return_correlation.png)

![12 Sector allocation](../reports/charts/eda/12_sector_allocation_donut.png)

![13 Top returns](../reports/charts/eda/13_top_3yr_returns.png)

![14 Expense vs return](../reports/charts/eda/14_expense_vs_return.png)

![15 Transaction trends](../reports/charts/eda/15_transaction_type_trends.png)